# AI-Native SOC: Network Intrusion ML Training

This notebook downloads the **CSE-CIC-IDS2018** Machine Learning CSV datasets from AWS S3, trains a Random Forest Anomaly Detection model, and exports the `.pkl` file for use in the local AI SOC backend.

### Instructions:
1. Click **Runtime** > **Run all**.
2. The notebook will automatically download the required data from AWS S3.
3. It will train the AI model on the traffic anomalies.
4. Finally, it will automatically prompt you to download `network_model.pkl`. Place this file in your local `backend/models/` folder.

In [ ]:
!pip install -q scikit-learn pandas numpy joblib awscli

### 1. Download Dataset from AWS S3
We use `--no-sign-request` to access the public dataset. To save time and RAM, we will download a single day of data containing web attacks and infiltration (e.g., Thursday).

In [ ]:
!mkdir -p data
!aws s3 cp --no-sign-request --region ca-central-1 "s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv" data/dataset.csv

### 2. Load and Preprocess Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib

print("Loading dataset (this may take a minute)...")
# Load a subset of rows to prevent Colab RAM crashes if the file is massive
df = pd.read_csv('data/dataset.csv', nrows=500000)

# Clean up column names (remove leading/trailing spaces)
df.columns = df.columns.str.strip()

print(f"Loaded {len(df)} rows.")
print("Labels found:", df['Label'].unique())

# Convert labels: Benign = 0, Malicious (anything else) = 1
df['is_malicious'] = (df['Label'] != 'Benign').astype(int)

# Select a subset of highly correlated features for a lightweight, fast model
features = [
    'Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Fwd Packet Length Max', 'Fwd Packet Length Mean', 'Bwd Packet Length Max', 
    'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s', 'Packet Length Mean',
    'Average Packet Size'
]

# Clean data (replace inf/-inf with NaN, then impute)
df.replace([np.inf, -np.inf], np.nan, inplace=True)

X = df[features]
y = df['is_malicious']

# Impute missing values with 0
imputer = SimpleImputer(strategy='constant', fill_value=0)
X = imputer.fit_transform(X)

# Scale the features
scaler = StandardScaler()
X = scaler.fit_transform(X)

print("Data preprocessed successfully.")

### 3. Train the Anomaly Detection Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Random Forest Classifier...")
clf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

accuracy = clf.score(X_test, y_test)
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")

### 4. Export the Model

In [ ]:
print("Saving model pipeline...")
model_pipeline = {
    'features': features,
    'imputer': imputer,
    'scaler': scaler,
    'classifier': clf
}

joblib.dump(model_pipeline, 'network_model.pkl')
print("Model saved as 'network_model.pkl'")

# Automatically download to your local machine
from google.colab import files
files.download('network_model.pkl')